# 03 - Delta mutations, MERGE, history, and recovery

Prerequisites / execution context: Run 00. Independent sandbox table; re-running resets only delta_lab.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R07: Delta Lake](https://docs.databricks.com/aws/en/delta/); [R26: Delta MERGE](https://docs.databricks.com/aws/en/delta/merge); [R27: Table history and time travel](https://docs.databricks.com/aws/en/tables/history); [R28: VACUUM](https://docs.databricks.com/aws/en/tables/operations/vacuum).


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 1. Create a small baseline and record its version
The sandbox isolates recovery experiments from the main Silver and Gold tables.


In [ ]:
T=f"{SILVER}.delta_lab"
spark.sql(f"CREATE OR REPLACE TABLE {T} (id INT, amount DECIMAL(12,2), seq INT) USING DELTA")
spark.sql(f"INSERT INTO {T} VALUES (1,10.00,1),(2,20.00,1)")
baseline=spark.sql(f"DESCRIBE HISTORY {T} LIMIT 1").first().version


## 2. Merge an update and an insert
The sequence predicate protects current values against an older incoming event. Deduplicate the source before MERGE when more than one source record can update the same target row. Official Databricks documentation: [R26: Delta MERGE](https://docs.databricks.com/aws/en/delta/merge).


In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW updates AS SELECT * FROM VALUES (1,15.00,2),(3,30.00,1) AS u(id,amount,seq)")
merge_sql=f"""MERGE INTO {T} t USING updates s ON t.id=s.id
WHEN MATCHED AND s.seq>t.seq THEN UPDATE SET amount=s.amount, seq=s.seq
WHEN NOT MATCHED THEN INSERT (id,amount,seq) VALUES (s.id,s.amount,s.seq)"""
spark.sql(merge_sql)
assert spark.table(T).count()==3
spark.sql(merge_sql)
assert spark.table(T).count()==3
assert str(spark.table(T).filter("id=1").first().amount)=="15.00"


## 3. Time travel versus restore
Time travel reads an earlier snapshot. RESTORE writes a new current version. It can cause downstream consumers to observe data changes again; it is not a replacement for backup. Official Databricks documentation: [R27: Table history and time travel](https://docs.databricks.com/aws/en/tables/history).


In [ ]:
assert spark.sql(f"SELECT * FROM {T} VERSION AS OF {baseline}").count()==2
spark.sql(f"UPDATE {T} SET amount=999 WHERE id=2")
spark.sql(f"DELETE FROM {T} WHERE id=3")
spark.sql(f"RESTORE TABLE {T} TO VERSION AS OF {baseline}")
assert spark.table(T).count()==2
spark.sql(f"DESCRIBE HISTORY {T}").show(truncate=False)
# Inspection only: no old files are deleted by DRY RUN.
spark.sql(f"VACUUM {T} DRY RUN").show(truncate=False)


## Challenge
Would deleting the transaction log repair this table? No. Use documented recovery operations. Why can an old version be listed but unreadable? Required data files may have expired and been vacuumed. Do not shorten retention to force this failure.
